In [ ]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import os
import asyncio
import requests
from agents import Agent, Runner, trace, function_tool
import gradio as gr
from pydantic import BaseModel


In [ ]:

load_dotenv(override=True)
openai = OpenAI()

In [ ]:
# For pushover

pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    print(f"Pushover user found and starts with {pushover_user[0]}")
else:
    print("Pushover user not found")

if pushover_token:
    print(f"Pushover token found and starts with {pushover_token[0]}")
else:
    print("Pushover token not found")

In [ ]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [ ]:
class PushInput(BaseModel):
    message: str

@function_tool
def push(input: PushInput) -> str:
    print(f"Push: {input.message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": input.message}
    requests.post(pushover_url, data=payload)
    return "Push notification sent."

In [ ]:
# Initialize Groceries dicts
groceries_inventory = {"eggs": 12, "milk": 16, "bread": 10, "bananas": 6}
groceries_consumed = {}
richConsole = True

In [ ]:
groceries_inventory

In [ ]:
@function_tool
def mark_complete(item: str):
    if item in groceries_inventory.keys():
        quantity = groceries_inventory[item]

        if quantity == 0:
            message = f"[red]Please order {item} at this time[/red]\n" if richConsole else f"Please order {item} at this time\n"
            push(message)
            show(message) if richConsole else print(message)
    else:
        print("Item not found in the grocery list")

In [ ]:
@function_tool
def update_groceries_status() -> str:
    remaining = {
        key: groceries_inventory[key] - groceries_consumed.get(key, 0)
        for key in groceries_inventory
    }

    result = ""
    for grocery, quantity in remaining.items():
        if quantity == 0:
            mark_complete(grocery)
            result += f"Grocery #{grocery}: [red]{quantity}[/red]\n" if richConsole else f"Grocery #{grocery}: {quantity}\n"
        else:
            result += f"Grocery #{grocery}: {quantity}\n"

    groceries_inventory.update(remaining)
    for key in remaining:
        groceries_consumed[key] = 0

    if richConsole:
        show(result)

    return result

In [ ]:
class ConsumedItems(BaseModel):
    items: list[str]
    quantities: list[int]

In [ ]:
@function_tool
def update_groceries_consumed(consumed: ConsumedItems) -> str:
    """
    Updates the inventory by recording consumed items.
    Args:
        consumed: An object with parallel lists of item names and quantities.
    """
    result_lines = []
    
    for grocery, quantity in zip(consumed.items, consumed.quantities):
        if grocery not in groceries_inventory:
            result_lines.append(f"Item '{grocery}' not found in the grocery list.")
            continue
        
        if quantity < 0:
            result_lines.append(f"Invalid quantity for {grocery}: {quantity}")
            continue
            
        available = groceries_inventory[grocery] - groceries_consumed.get(grocery, 0)
        if quantity > available:
            result_lines.append(f"You cannot consume more {grocery} than available ({available}).")
            continue
            
        groceries_consumed[grocery] = groceries_consumed.get(grocery, 0) + quantity
        result_lines.append(f"Successfully updated {grocery}.")

    final_report = "\n".join(result_lines)
    
    if richConsole:
        print(final_report) 
        
    return final_report

In [ ]:
class GroceryItem(BaseModel):
    name: str
    quantity: int

In [ ]:
@function_tool
def create_groceries_inventory(inventory: list[GroceryItem]) -> str:
    for item in inventory:
        groceries_inventory[item.name] = item.quantity
    result = update_groceries_status()
    return result

In [ ]:
create_groceries_inventory({"eggs": 10, "milk": 16, "bread": 10, "bananas": 6})

In [ ]:
tools = [create_groceries_inventory, mark_complete, update_groceries_status, update_groceries_consumed, push]

In [ ]:
instructions = """
You are a smart refrigerator assistant. You manage the user's grocery inventory and consumption.

**Interpreting the user:** Parse item names and quantities from natural or shorthand phrasing. All of these mean the same thing and should be interpreted as one item with a quantity:
- "I have 10 eggs" / "10 eggs" / "eggs 10" / "eggs: 10" → eggs: 10
- "I used 2 milk" / "milk 2" / "2 cups milk" → milk: 2 (use the number given; treat "cups" or "loaves" as the unit, quantity is the number)
Normalize item names to simple lowercase words (e.g. eggs, milk, bread, bananas) for tool calls. If the user says "I have 10 eggs" when setting up, pass {"eggs": 10}; if they say "eggs 10" when reporting consumption, pass {"eggs": 10} to update_groceries_consumed.

**Setup:** When the user gives a starting inventory (in any of the phrasings above), call create_groceries_inventory with a single object mapping item names to quantities (integers).

**When the user reports consumption:** Call update_groceries_consumed with an object of item names to quantities consumed (interpret "I used X", "X 3", "3 X", etc. as that item and quantity). Then call update_groceries_status.

**When any item's remaining quantity is zero:** Call mark_complete for that item so the user is notified to reorder it.

**Rules:** Use only the tools above; do not invent tools. If the user does not specify a quantity, infer a reasonable amount or use 0. Reply in Rich console markup where helpful; no code blocks. Do not ask for clarification—use your tools and then respond with a clear summary or status.
"""

In [ ]:

smart_refrigerator = Agent(name="Smart Refigerator", instructions=instructions, tools=tools, model="gpt-4o-mini")

message = "I'll tell you when I use or consume some of the groceries. Please tell me when I need to reorder any item (i.e. when something runs out)"

In [ ]:
async def chat(message, history):
    # Strip any extra fields (like 'metadata') that Gradio adds
    clean_history = [{"role": m["role"], "content": m["content"]} for m in history]
    messages = clean_history + [{"role": "user", "content": message}]
    
    with trace("Groceries Inventory"):
        result = await Runner.run(smart_refrigerator, messages)
        return result.final_output

In [ ]:
richConsole = True
groceries_inventory = {}
groceries_consumed = {}
gr.ChatInterface(
    fn=chat,
    title="Smart Refrigerator Agent",
    type="messages",
    save_history=True,
    description="Ask about your groceries inventory, and the agent will respond."
).launch()